<a href="https://colab.research.google.com/github/bsheese/387/blob/main/23_language/23_1_Attention/23_1_1_Attention_by_Hand.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Language: Part 3
## Attention by Hand

---

## What This Notebook Is About

At the end of 23_0_2, DistilBERT turned the single table vector for "bank" into different vectors depending on whether the sentence was about money or a river. Each token's vector had taken in information from the other tokens around it. The mechanism that does this is **attention**, and it's the core of the **transformer**, the kind of network behind DistilBERT, GPT-2 and today's large language models.

Attention is built from three things you already know:

- the **dot product** as a measure of similarity (19_1_3, 23_0_2)
- **softmax**, which turns scores into weights that add up to 1 (19_1_4)
- a **weighted average**

This notebook builds it by hand on a few tokens with tiny vectors, adds the pieces real transformers use, checks the result against PyTorch, and then looks at attention weights inside DistilBERT.

**What you will learn:**
1. Attention as a weighted average whose weights come from dot products and softmax
2. Doing it for every token at once with matrix multiplication
3. Queries, keys and values, and why the scores are divided by $\sqrt{d}$
4. Masks: ignoring padding, and not looking ahead
5. How attention fits into a transformer, and why long texts are expensive
6. What DistilBERT's attention weights do and don't tell you

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

np.set_printoptions(precision=3, suppress=True)

def softmax(scores):
    # 19_1_4's softmax, applied to each row
    exps = np.exp(scores - scores.max(axis=-1, keepdims=True))
    return exps / exps.sum(axis=-1, keepdims=True)

(Subtracting each row's largest score before `np.exp` doesn't change the result, since softmax only depends on differences between scores (19_1_4). It just keeps `np.exp` from overflowing on large scores.)

---

## Section 1: One Token, by Hand

Take the four tokens "river", "bank", "money", "loan" with made-up embeddings of just 4 numbers each. The numbers are chosen so that "river" points one way, "money" and "loan" point another way, and "bank" sits in between:

In [ ]:
tokens = ['river', 'bank', 'money', 'loan']
X = np.array([[1.0, 0.9, 0.0, 0.1],    # river
              [0.6, 0.5, 0.6, 0.5],    # bank: a bit of both
              [0.0, 0.1, 1.0, 0.9],    # money
              [0.1, 0.0, 0.9, 1.0]])   # loan
print(X.shape)

To compute a new, context-aware vector for one token, attention does three steps.

**1. Scores.** Take the dot product of that token's vector with every token's vector, including its own. A high score means "similar to me".

In [ ]:
i = tokens.index('money')
scores = X @ X[i]
print('scores for "money":', dict(zip(tokens, scores.round(2))))

**2. Weights.** Turn the scores into weights with softmax, so they're all positive and add up to 1.

In [ ]:
weights = softmax(scores)
print('weights for "money":', dict(zip(tokens, weights.round(3))), ' total', weights.sum().round(3))

**3. Weighted average.** The new vector for "money" is the weighted average of all the token vectors: each vector multiplied by its weight, then added up.

In [ ]:
new_money = weights @ X        # same as weights[0]*X[0] + weights[1]*X[1] + ...
print('old "money":', X[i])
print('new "money":', new_money)

The new vector for "money" is mostly made of "money" and "loan", the tokens most similar to it, with a little of "bank" and very little of "river". That's all attention is: each token's new vector is an average of the tokens' vectors, weighted toward the ones most similar to it. The name comes from the weights: they say how much each token "attends to" each other token.

---

## Section 2: Every Token at Once

Doing this for every token is a matrix multiplication (19_1_3). `X @ X.T` gives every token's dot product with every other token, a 4 × 4 table of scores with one row per token. Softmax across each row turns it into weights, and multiplying the weights by `X` gives every token's weighted average.

In [ ]:
scores = X @ X.T                 # (4, 4): row i holds token i's scores for every token
weights = softmax(scores)        # each row adds up to 1
output = weights @ X             # (4, 4) @ (4, 4): row i is token i's weighted average

def show_weights(weights, rows, cols, title):
    plt.figure(figsize=(4.5, 3.8))
    plt.imshow(weights, cmap='Blues', vmin=0, vmax=1)
    plt.xticks(range(len(cols)), cols); plt.yticks(range(len(rows)), rows)
    for r in range(len(rows)):
        for c in range(len(cols)):
            plt.text(c, r, f'{weights[r, c]:.2f}', ha='center', va='center', fontsize=9)
    plt.xlabel('attends to'); plt.title(title); plt.colorbar(); plt.show()

show_weights(weights, tokens, tokens, 'attention weights')
print('new vectors:\n', output)

Read the table by rows: row "bank" shows how the new "bank" vector is built. Because "bank" is in between, it draws on all four tokens, while "river" draws mostly on itself and "money" and "loan" mostly on each other. In a real sentence the effect is what 23_0_2 measured: "bank" in a sentence with "loan" moves toward "loan", and "bank" in a sentence with "river" moves toward "river".

The shapes follow 19_1_3's rule. With $n$ tokens and vectors of length $d$, the scores are $(n, d) @ (d, n) = (n, n)$ and the output is $(n, n) @ (n, d) = (n, d)$: one new vector per token, the same length as before.

---

## Section 3: Queries, Keys and Values

Plain dot products of the embeddings have two problems. Every token is most similar to itself, so it mostly attends to itself. And one vector has to do three different jobs: describe what this token is looking for, describe what it offers to others, and carry the information that gets passed along. Real attention gives each job its own learned vector, made by three `nn.Linear` layers (without biases) applied to every token:

- the **query**: what this token is looking for
- the **key**: what this token offers to be matched against
- the **value**: what this token passes on if it's attended to

The scores are each token's query dotted with every token's key, and the weighted average is taken over the values. A token can then attend strongly to others that don't look like it, as long as its query matches their keys. The three weight matrices are learned by gradient descent, like any other weights.

In [ ]:
torch.manual_seed(0)
d = 4
W_q, W_k, W_v = nn.Linear(d, d, bias=False), nn.Linear(d, d, bias=False), nn.Linear(d, d, bias=False)

Xt = torch.tensor(X, dtype=torch.float32)
with torch.no_grad():
    Q, K, V = W_q(Xt), W_k(Xt), W_v(Xt)        # each (4, 4): one query, key and value per token
    scores = Q @ K.T / np.sqrt(d)              # Section 3's scaling, explained next
    weights = scores.softmax(dim=1)
    output = weights @ V
print('Q, K, V shapes:', tuple(Q.shape), tuple(K.shape), tuple(V.shape))
show_weights(weights.numpy(), tokens, tokens, 'with random query/key weights')

With random, untrained matrices the weights are meaningless, as you'd expect; training is what makes them useful. The calculation is complete, though. This is **scaled dot-product attention**:

$$\text{attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^{T}}{\sqrt{d}}\right) V$$

### Why divide by $\sqrt{d}$?

The dot product adds up $d$ products, so the longer the vectors, the larger the scores tend to be. 19_1_4 showed that multiplying scores by a larger number makes softmax more extreme, with one weight near 1 and the rest near 0. With long vectors, softmax would almost always pick out a single token, and its slopes would be tiny (when softmax is that extreme, small changes to the scores barely change its output), so training would stall. Dividing by $\sqrt{d}$ keeps the scores on the same scale whatever the length:

In [ ]:
torch.manual_seed(0)
for d in [4, 64, 512]:
    q = torch.randn(2000, d, 1)                         # 2,000 random queries...
    k = torch.randn(2000, 10, d)                        # ...each with 10 random keys
    raw = (k @ q).squeeze(-1)                           # (2000, 10) scores
    scaled = raw / d ** 0.5
    print(f'd = {d:3}: spread of scores {raw.std():5.1f}, average largest weight {raw.softmax(1).max(1).values.mean():.2f}   |   '
          f'divided by sqrt(d): spread {scaled.std():.1f}, average largest weight {scaled.softmax(1).max(1).values.mean():.2f}')

Each line averages over 2,000 random queries, each compared with 10 random keys. Without the division, the spread of the scores grows with $d$, and by $d = 512$ softmax puts almost all of its weight on one key. With it, the spread stays at about 1 and the largest weight stays the same whatever $d$ is. DistilBERT uses $d = 64$ per head (Section 6).

### PyTorch does the same thing

PyTorch has this calculation built in as `torch.nn.functional.scaled_dot_product_attention`:

In [ ]:
builtin = nn.functional.scaled_dot_product_attention(Q.unsqueeze(0), K.unsqueeze(0), V.unsqueeze(0))[0]
print('same as by hand:', torch.allclose(builtin, output, atol=1e-6))

(`.unsqueeze(0)` adds a batch dimension, as in 20_0_1.)

---

## Section 4: Masks

Sometimes a token must not attend to certain others. The way to stop it is to set those scores to $-\infty$ before softmax: $e^{-\infty} = 0$, so those weights become exactly 0 and the rest still add up to 1.

**Padding.** In a batch, short texts are padded (23_0_1), and real tokens shouldn't take in information from `[PAD]` tokens. The attention mask from 23_0_1 marks which scores to block.

**Looking ahead.** A model that generates text, like GPT-2 in 23_3, predicts each next token from the ones before it. During training it sees whole texts, so it has to be stopped from cheating by attending to tokens that come later. A **causal mask** blocks every score above the diagonal: each token can attend only to itself and the tokens before it.

In [ ]:
sentence = ['the', 'movie', 'was', 'not', 'good']
torch.manual_seed(1)
E = torch.randn(5, 8)
scores = E @ E.T / np.sqrt(8)

causal = torch.triu(torch.ones(5, 5, dtype=torch.bool), diagonal=1)     # True above the diagonal
masked = scores.masked_fill(causal, float('-inf'))
show_weights(masked.softmax(dim=1).numpy(), sentence, sentence, 'with a causal mask')

The top-right triangle is all zeros. "the" can only attend to itself, "movie" to "the" and itself, and so on, and every row still adds up to 1. Notice that "good" is processed before the model knows anything after it, which is one reason text generators work one token at a time.

---

## Section 5: Attention Ignores Order Unless It's Told

Shuffle the tokens, and the attention output is just shuffled the same way: nothing about a token's new vector depends on *where* the other tokens are, only on *what* they are. "Dog bites man" and "man bites dog" would give each word the same new vector.

In [ ]:
order = [2, 0, 3, 1]
shuffled_out = softmax(X[order] @ X[order].T) @ X[order]
original_out = softmax(X @ X.T) @ X
print('shuffled output equals the original output, shuffled:', np.allclose(shuffled_out, original_out[order]))

Transformers fix this by adding a second embedding to each token's vector before the first layer: a **position embedding**, one learned vector for each position (first token, second token, ...). DistilBERT has 512 of them, which is why it can't read more than 512 tokens. After that, the same word in different positions starts from different vectors, and attention can use order.

---

## Section 6: Attention Inside a Transformer

A real transformer layer adds a few things around the attention you just built:

- **Several heads.** Instead of one attention calculation on the full vectors, a layer runs several smaller ones side by side, each with its own query, key and value matrices, so that different heads can attend for different reasons. DistilBERT has 12 heads per layer, each using vectors of 64 numbers (12 × 64 = 768), and their outputs are joined back together and passed through one more `nn.Linear`.
- **A feed-forward network** after attention: two `nn.Linear` layers with an activation between them, applied to each token's vector separately, like a small MLP from unit 21.
- **Residual connections** around both, as in ResNet (22_1_1), and **layer normalization**, which standardizes each token's vector the way batch norm standardizes each channel (22_0_2).

Here are the parts of one of DistilBERT's six layers:

In [ ]:
from transformers import AutoModel, AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
distilbert = AutoModel.from_pretrained('distilbert-base-uncased', attn_implementation='eager').eval()
print(distilbert.transformer.layer[0])

`q_lin`, `k_lin` and `v_lin` are the query, key and value layers, each 768 → 768 (all 12 heads together), and `out_lin` combines the heads. `ffn` is the feed-forward network, and the two `LayerNorm`s are the layer normalizations. DistilBERT stacks six of these. The output of the sixth is the contextual embeddings from 23_0_2.

### Why long texts are expensive

The score table has one entry for every pair of tokens: $n \times n$ for $n$ tokens, in every head of every layer. Doubling the length of a text doubles the work for most of a transformer, but quadruples the attention scores:

In [ ]:
for n in [128, 256, 512, 1024]:
    print(f'{n:5} tokens: {n * n:>9,} scores per head, {n * n * 12 * 6:>12,} in all of DistilBERT')

This is why 23_0_1 truncated reviews, and why a long context window is one of the most expensive features of large language models.

---

## Section 7: What DistilBERT's Attention Weights Show

Since every layer's attention weights can be read out, it's tempting to treat them as an explanation of what the model is "thinking". Here's a test. In "The animal didn't cross the street because it was too tired", "it" means the animal. Change "tired" to "wide" and "it" means the street. Does attention from "it" follow the meaning? The next cell shows, for both sentences, the attention weights from "it" in DistilBERT's fifth layer, averaged over its 12 heads.

In [ ]:
def attention_from(sentence, word, layer):
    encoded = tokenizer(sentence, return_tensors='pt')
    toks = tokenizer.convert_ids_to_tokens(encoded['input_ids'][0])
    with torch.no_grad():
        attentions = distilbert(**encoded, output_attentions=True).attentions   # one per layer: (1, heads, n, n)
    return toks, attentions[layer][0, :, toks.index(word)].mean(dim=0)           # average over heads

fig, axes = plt.subplots(2, 1, figsize=(10, 5.5))
for ax, ending in zip(axes, ['tired', 'wide']):
    sentence = f"The animal didn't cross the street because it was too {ending}."
    toks, w = attention_from(sentence, 'it', layer=4)
    ax.bar(range(len(toks)), w)
    ax.set_xticks(range(len(toks)), toks)
    ax.set_title(f'..."because it was too {ending}": attention from "it" (layer 5, average of 12 heads)', fontsize=10)
    ax.set_ylabel('weight')
    print(f"{ending:5}  animal {w[toks.index('animal')]:.2f}   street {w[toks.index('street')]:.2f}   "
          f"[SEP] {w[toks.index('[SEP]')]:.2f}")
plt.tight_layout()
plt.show()

In both sentences, "it" attends much more to "animal" than to "street", even when "it" means the street. A lot of the weight also goes to `[SEP]` and to the final period, tokens that carry no meaning about animals or streets at all. Researchers studying BERT, the larger model DistilBERT was made from, found that its heads often put their weight on `[SEP]` and punctuation when they have nothing in particular to look at (Clark et al., 2019).

That doesn't mean DistilBERT can't work out what "it" refers to; the information may be combined across many heads, layers and the feed-forward networks in ways no single attention map shows. It means attention weights tell you where one step of the calculation drew information from, not why the model gives its final answer. Pictures of attention neatly linking a word like "it" to the right noun usually show one carefully chosen head. Reading attention weights as explanations calls for the same caution as reading feature maps in 22_1_1.

[Transformer Explainer](https://poloclub.github.io/transformer-explainer/) runs GPT-2 in your browser and shows its queries, keys, values and attention weights for any text you type, which makes all of the steps in this notebook visible on a real model.

---

## Summary

- **Attention** gives each token a new vector that's a **weighted average** of the tokens' vectors. The weights come from **dot products** (similarity) passed through **softmax**, so each row of weights adds up to 1.
- For all tokens at once: scores $(n, n)$, softmax across each row, then weights $(n, n)$ times vectors $(n, d)$.
- Real attention uses learned **queries** (what a token looks for), **keys** (what it offers) and **values** (what it passes on): $\text{softmax}(QK^T / \sqrt{d})\,V$. Dividing by $\sqrt{d}$ stops long vectors from making softmax pick a single token. PyTorch's `scaled_dot_product_attention` computes the same thing.
- **Masks** set blocked scores to $-\infty$ so their weights are 0: padding masks ignore `[PAD]` tokens, and **causal masks** stop a text generator from looking ahead.
- Attention alone ignores word order; **position embeddings** add it, and limit DistilBERT to 512 tokens.
- A **transformer** layer combines several attention **heads** with a feed-forward network, residual connections and layer normalization. DistilBERT has 6 layers of 12 heads. Attention's cost grows with the square of the text length.
- DistilBERT's attention from "it" pointed to "animal" whether "it" meant the animal or the street, and much of it went to `[SEP]` and the period. Attention weights show where information was drawn from, not an explanation of the answer.

**Next:** 23_2 uses pretrained transformers: first ready-made models for common tasks, then fine-tuning DistilBERT to classify the IMDB reviews, the text version of 22_1's transfer learning.